# Solar Filament Segmentation 2026 - Official Reproduction Notebook

This self-contained notebook reproduces the final submission () end-to-end:
1. Auto-detects compute hardware (CUDA GPU or CPU).
2. Runs unified pipeline inference via  on Config A (or Config B).
3. Validates the submission CSV format, dimensions, non-emptiness, and disjointness.
4. Compares predictions against  asserting match fraction > 0.999 at IoU > 0.9.
5. Reports total and per-image runtime.

In [ ]:
# 1. Environment & Dependencies Setup
import os
import sys
import time
from pathlib import Path

# Single-thread CPU safety for reproducible BLAS ops
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import torch
device_name = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Execution Device: {device_name.upper()}")


In [ ]:
# 2. Reproduction Parameters
# Test images directory parameter (defaults to Kaggle input or local workspace)
kaggle_test_dir = Path("/kaggle/input/filament-segmentation-2026/test/test_images")
local_test_dir = Path("MAGFiLO_1.0_Kaggle_2026/test/test_images")
if kaggle_test_dir.exists():
    TEST_IMAGES_DIR = kaggle_test_dir
elif local_test_dir.exists():
    TEST_IMAGES_DIR = local_test_dir
else:
    TEST_IMAGES_DIR = Path("test_images")

# Configuration Selection:
# False -> Config A (R1b Rule: tm=0.55, cr=3, tp=0.75, ma=200) reproducing submission_R1b.csv
# True  -> Config B (Plateau Rule: tm=0.60, cr=2, tp=0.75, ma=300)
USE_CONFIG_B = False

# Quick smoke test mode (5 images only)
QUICK = False

OUTPUT_CSV = Path("submission.csv")
CONFIG_PATH = Path("configs/final_B.yaml" if USE_CONFIG_B else "configs/final_A.yaml")
EXPECTED_CSV = Path("expected/submission_R1b.csv")

print(f"Target Config: {CONFIG_PATH}")
print(f"Test Images Directory: {TEST_IMAGES_DIR}")
print(f"Quick Mode (5 images): {QUICK}")


In [ ]:
# 3. Run Pipeline Inference via solarseg.run
from solarseg.run import run_pipeline

limit_n = 5 if QUICK else None
print(f"Starting inference pipeline (limit={limit_n})...")
t0 = time.time()

run_pipeline(
    config_path=CONFIG_PATH,
    images_dir=TEST_IMAGES_DIR,
    output_csv=OUTPUT_CSV,
    ensemble=1,
    limit=limit_n,
    device_str=device_name,
)

total_time = time.time() - t0
n_images = len(list(TEST_IMAGES_DIR.glob("*"))) if not QUICK else 5
per_image_time = total_time / max(n_images, 1)

print("=" * 50)
print(f"Inference Completed Successfully!")
print(f"Total Runtime:      {total_time:.1f} s ({total_time / 60.0:.2f} min)")
print(f"Per-Image Runtime:  {per_image_time:.3f} s/image")
print("=" * 50)


In [ ]:
# 4. Submission Validation (Hard Rules Verification)
from scripts.check_submission import verify_submission_file

print(f"Verifying generated submission: {OUTPUT_CSV}...")
is_valid = verify_submission_file(OUTPUT_CSV, test_images_dir=TEST_IMAGES_DIR)
assert is_valid, "Submission failed verification!"


In [ ]:
# 5. Mask Matching Verification against Reference (IoU > 0.9 Assertion)
import csv
import numpy as np
from solarseg.rle import decode_rle_to_mask

if not USE_CONFIG_B and not QUICK and EXPECTED_CSV.exists():
    print(f"Evaluating mask agreement against expected reference: {EXPECTED_CSV}...")
    def load_masks(csv_p):
        masks_by_img = {}
        with open(csv_p, "r") as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if len(row) == 2:
                    stem = row[0].rsplit("_", 1)[0]
                    masks_by_img.setdefault(stem, []).append(row[1])
        return masks_by_img

    gen_masks = load_masks(OUTPUT_CSV)
    ref_masks = load_masks(EXPECTED_CSV)

    total_ref_instances = 0
    matched_instances = 0

    for stem, rles_ref in ref_masks.items():
        rles_gen = gen_masks.get(stem, [])
        masks_r = [decode_rle_to_mask(r, (2048, 2048)) for r in rles_ref]
        masks_g = [decode_rle_to_mask(g, (2048, 2048)) for g in rles_gen]
        total_ref_instances += len(masks_r)

        for mr in masks_r:
            matched = False
            for mg in masks_g:
                inter = np.count_nonzero(mr & mg)
                union = np.count_nonzero(mr | mg)
                iou = inter / union if union > 0 else 0.0
                if iou >= 0.9:
                    matched = True
                    break
            if matched:
                matched_instances += 1

    match_fraction = matched_instances / max(total_ref_instances, 1)
    print(f"Reference Instances: {total_ref_instances}")
    print(f"Matched Instances (IoU >= 0.9): {matched_instances}")
    print(f"Match Fraction: {match_fraction:.6f} ({match_fraction*100.0:.2f}%)")
    assert match_fraction >= 0.999, f"Match fraction {match_fraction:.4f} is below mandated threshold 0.999!"
    print("[PASS] Mask match assertion passed (> 0.999)!")
else:
    print("[INFO] Skipped reference matching (either USE_CONFIG_B=True or QUICK=True or expected file missing).")
